[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/divyamohan1993/dip-practical/blob/main/Practical_1.ipynb)

> **Run in Google Colab:** Click the badge above to open this notebook in Google Colab. The dataset downloads automatically — no setup needed.

<style>
/* =====================================================================
   Practical Handbook — Uniform Print Formatting (CSU2543 Digital Image Processing)
   Sections are color-coded: Aim (blue), Theory (purple), Code (green),
   Output (amber), Analysis (maroon). Code blocks are uniform across all
   practicals. Page layout: 1-inch margins, justified text, Times New Roman.
   ===================================================================== */
@page { margin: 1in; }
@media print {
  body, .jp-Notebook, .jupyter-renderer { background: white !important; }
  .jp-CodeMirrorEditor, .CodeMirror { font-size: 10pt !important; }
  .jp-Cell-inputCollapser, .jp-Cell-outputCollapser, .jp-Toolbar { display: none !important; }
  .dip-section-marker { page-break-inside: avoid; }
}

.jp-RenderedHTMLCommon, .jp-RenderedMarkdown {
  font-family: 'Times New Roman', Georgia, serif;
  font-size: 12pt;
  line-height: 1.55;
  text-align: justify;
}

/* Section markers — colored heading bars */
.dip-section-marker {
  display: block;
  font-weight: bold;
  font-size: 1.35em;
  padding: 0.45em 0.7em;
  margin: 1.1em 0 0.6em 0;
  border-left: 6px solid;
  border-radius: 3px;
  letter-spacing: 0.02em;
  page-break-after: avoid;
}
.dip-section-aim      { color: #003c8f; background: #e3f2fd; border-color: #1565c0; }
.dip-section-theory   { color: #4a148c; background: #f3e5f5; border-color: #6a1b9a; }
.dip-section-code     { color: #1b5e20; background: #e8f5e9; border-color: #2e7d32; }
.dip-section-output   { color: #e65100; background: #fff3e0; border-color: #ef6c00; }
.dip-section-analysis { color: #b71c1c; background: #ffebee; border-color: #c62828; }

/* Sub-section headings within a section ("Part 1", "Part 2", ...) */
.dip-subsection {
  font-weight: 600;
  font-size: 1.1em;
  margin: 0.9em 0 0.4em 0;
  color: #2e7d32;
  border-bottom: 1px solid #c8e6c9;
  padding-bottom: 0.15em;
  page-break-after: avoid;
}

/* Code cells — uniform JetBrains Mono / Source Code Pro across all practicals */
.jp-CodeMirrorEditor, .jp-CodeCell .jp-InputArea-editor,
.CodeMirror, pre, .highlight, code {
  font-family: 'JetBrains Mono', 'Source Code Pro', Consolas, 'Courier New', monospace !important;
}
.jp-CodeCell .jp-InputArea-editor, .CodeMirror, pre {
  font-size: 10pt !important;
  background: #f6f8fa !important;
  border-left: 3px solid #2e7d32 !important;
  border-radius: 0 !important;
  padding: 8px 12px !important;
}
code { background: #f6f8fa; padding: 1px 5px; border-radius: 2px; font-size: 0.95em; }

/* Output cells — amber tint, matching the Output section colour */
.jp-OutputArea-output {
  border-left: 3px solid #ef6c00 !important;
  background: #fffaf3 !important;
  padding: 6px 10px !important;
}

/* Analysis question lists */
.dip-analysis-list { padding-left: 1.4em; }
.dip-analysis-list li { margin-bottom: 0.5em; text-align: justify; }
</style>

# Practical 1: Loading and Displaying Digital Images

<span class="dip-section-marker dip-section-aim">1. Aim</span>

To learn how to load, display, and inspect digital images using OpenCV and Matplotlib, and to characterise their basic properties (shape, intensity range, mean, standard deviation) and pixel-intensity distribution via the histogram.

<span class="dip-section-marker dip-section-theory">2. Description / Theory</span>

A digital image is the discrete sampled representation of a continuous two-dimensional intensity function $f(x,y)$ on a regular spatial grid. After sampling on an $M\times N$ grid and quantising the amplitude to $L$ intensity levels, the image is stored as an $M\times N$ matrix of unsigned integers. For an 8-bit grayscale image $L=256$, so each pixel lies in $[0,255]$ with $0$ encoding pure black and $255$ encoding pure white.

**Histogram.** The histogram $h(r_k)$ counts the number of pixels with intensity $r_k$ for $k=0,1,\dots,L-1$. The shape of the histogram captures the global tonal characteristics of the image: a narrow cluster indicates low contrast, a wide spread indicates high contrast, and bimodality often signals a foreground/background separation.

<span class="dip-section-marker dip-section-code">3. Code</span>

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install opencv-python-headless matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os


<span class="dip-subsection">Part 1: Image Loading</span>
Load a grayscale image using `cv2.imread()` and inspect its properties.


In [ ]:
# === AUTO-DOWNLOAD DATASET (works in Google Colab and locally) ===
import os, urllib.request, zipfile

# Choose which chapter to download (CH01-CH12)
CHAPTER = "CH02"  # Chapter 2: Digital Image Fundamentals
DATASET_PATH = f"datasets/{CHAPTER}/"
DOWNLOAD_BASE = "https://www.imageprocessingplace.com/downloads_V3/dip3e_downloads/dip3e_book_images"

if not os.path.exists(DATASET_PATH) or not any(f.endswith('.tif') for f in os.listdir(DATASET_PATH)):
    zip_name = f"DIP3E_{CHAPTER}_Original_Images.zip"
    url = f"{DOWNLOAD_BASE}/{zip_name}"
    print(f"Downloading {CHAPTER} dataset from imageprocessingplace.com...")
    urllib.request.urlretrieve(url, "chapter.zip")
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile("chapter.zip", "r") as z:
        for f in z.namelist():
            if f.lower().endswith(".tif"):
                fname = os.path.basename(f)
                if fname:
                    with z.open(f) as src, open(os.path.join(DATASET_PATH, fname), "wb") as dst:
                        dst.write(src.read())
    os.remove("chapter.zip")
    print(f"Downloaded {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")
else:
    print(f"Dataset ready: {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")

# List all available images
images = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])
print(f"\nAvailable images ({len(images)}):")
for i, name in enumerate(images, 1):
    print(f"  {i}. {name}")

In [ ]:
# === SELECT YOUR IMAGE HERE ===
selected_image = "Fig0222(b)(cameraman).tif"  # Change this to any image from the list above

# Load the image in grayscale
img = cv2.imread(os.path.join(DATASET_PATH, selected_image), cv2.IMREAD_GRAYSCALE)

# Display image properties
print(f"Image: {selected_image}")
print(f"Shape: {img.shape}")
print(f"Data type: {img.dtype}")
print(f"Min pixel value: {img.min()}")
print(f"Max pixel value: {img.max()}")
print(f"Mean pixel value: {img.mean():.2f}")


<span class="dip-subsection">Part 2: Display the Image</span>
Use `plt.imshow()` with `cmap='gray'` to display the grayscale image.


In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray')
plt.title(f"{selected_image} ({img.shape[0]}x{img.shape[1]})")
plt.axis('off')
plt.tight_layout()
plt.show()


<span class="dip-subsection">Part 3: Display Multiple Images</span>
Load and display 4 different images in a 2x2 grid.


In [ ]:
# === SELECT 4 IMAGES TO COMPARE ===
image_names = [
    "Fig0222(b)(cameraman).tif",
    "Fig0221(a)(ctskull-256).tif",
    "Fig0230(a)(dental_xray).tif",
    "Fig0226(galaxy_pair_original).tif"
]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, name in zip(axes.flatten(), image_names):
    image = cv2.imread(os.path.join(DATASET_PATH, name), cv2.IMREAD_GRAYSCALE)
    ax.imshow(image, cmap='gray')
    ax.set_title(f"{name.split('(')[-1].replace(').tif','')}\n{image.shape[0]}x{image.shape[1]}")
    ax.axis('off')
plt.suptitle("Multiple Images from Dataset", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


<span class="dip-subsection">Part 4: Image Properties and Histogram</span>
Inspect detailed properties and visualize the pixel intensity distribution.


In [ ]:
# Detailed properties
print(f"{'Property':<20} {'Value'}")
print("-" * 40)
print(f"{'Filename':<20} {selected_image}")
print(f"{'Dimensions':<20} {img.shape[0]} x {img.shape[1]}")
print(f"{'Total pixels':<20} {img.size}")
print(f"{'Data type':<20} {img.dtype}")
print(f"{'Min intensity':<20} {img.min()}")
print(f"{'Max intensity':<20} {img.max()}")
print(f"{'Mean intensity':<20} {img.mean():.2f}")
print(f"{'Std deviation':<20} {img.std():.2f}")


In [ ]:
# Display image alongside its histogram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(img, cmap='gray')
ax1.set_title(f"{selected_image}")
ax1.axis('off')

ax2.hist(img.ravel(), bins=256, range=(0, 256), color='black', alpha=0.7)
ax2.set_title("Pixel Intensity Histogram")
ax2.set_xlabel("Intensity Value (0-255)")
ax2.set_ylabel("Frequency")
ax2.set_xlim(0, 256)

plt.tight_layout()
plt.show()


<span class="dip-section-marker dip-section-output">4. Output</span>

*All output (printed values, computed statistics, and rendered figures) appears immediately below the corresponding code cells when this notebook is executed top-to-bottom.*

<span class="dip-section-marker dip-section-analysis">5. Analysis / Conclusion</span>

*Refer to the practical handbook for analysis questions and discussion.*